# Run full workers on Modal

Use one installed command operation locally and on Modal, inspect its accepted
outputs, and keep worker packing distinct from per-artifact command invocation.
The worker runs the complete preprocess, execute, and postprocess lifecycle.
No GPU is required.

This optional tutorial makes billable Modal and object-storage calls. Install
matching Artisan and `artisan-modal` packages, configure Modal authentication,
and supply a canonical Pixi manifest and lock containing both exact packages.
Build a checkout wheel with `python -m pip wheel --no-deps .` in an isolated
build frontend; the remote project also needs the matching provider wheel.
Do not point a remote worker at an editable notebook directory.

Set `ARTISAN_MODAL_WORKER_MANIFEST` to that project's absolute manifest path.
Its `default` environment must contain the matching packages and the Conda
`bash`, `coreutils`, and `inetutils` dependencies. `WaitTool` uses this environment
for Bash and the `seq`, `sleep`, `basename`, and `hostname` commands. Each notebook
uses a unique application name so preparation does not replace another run.
The environment cache follows `XDG_CACHE_HOME` (or `~/.cache`); set that standard
variable to an absolute writable directory when you need an isolated cache.
Set `ARTISAN_S3_BUCKET`, `ARTISAN_S3_ENDPOINT_URL`, `AWS_ACCESS_KEY_ID`, and
`AWS_SECRET_ACCESS_KEY` to a dedicated writable tutorial bucket/prefix. The
credentials are explicit runtime storage configuration for the controller and
worker; they do not belong in a wheel, Pixi lock, or scientific parameters.
The example uses an HTTPS endpoint. Use a narrowly scoped credential and keep
it out of notebook output. Missing setup raises an error before preparation.

Preparation is explicit and may build images and deploy named functions. Merely
constructing or selecting a runner does not provision software. Provider startup,
packing, and cancellation require qualification against the actual deployed
version; local examples alone do not establish those properties.

In [ ]:
from __future__ import annotations

import os
import shutil
from pathlib import Path
from tempfile import mkdtemp
from uuid import uuid4

from artisan_modal import ModalRunner

from artisan.schemas import Environment, StorageConfig

# This must be a locked project containing matching Artisan and provider wheels.
manifest = (
    Path(os.environ["ARTISAN_MODAL_WORKER_MANIFEST"]).expanduser().resolve(strict=True)
)
bucket = os.environ["ARTISAN_S3_BUCKET"]
endpoint = os.environ["ARTISAN_S3_ENDPOINT_URL"]
key = os.environ["AWS_ACCESS_KEY_ID"]
secret = os.environ["AWS_SECRET_ACCESS_KEY"]
region = os.environ.get("AWS_DEFAULT_REGION", "auto")
storage = StorageConfig(
    protocol="s3",
    options={
        "key": key,
        "secret": secret,
        "client_kwargs": {"endpoint_url": endpoint, "region_name": region},
    },
    delta_options={
        "AWS_ACCESS_KEY_ID": key,
        "AWS_SECRET_ACCESS_KEY": secret,
        "AWS_ENDPOINT_URL": endpoint,
        "AWS_REGION": region,
    },
)
workspace = Path(mkdtemp(prefix="artisan-modal-tutorial-"))
cache_root = (
    Path(os.environ.get("XDG_CACHE_HOME", Path.home() / ".cache"))
    / "artisan/environments"
)
app_name = f"artisan-tutorial-{uuid4().hex[:12]}"
prefix = f"s3://{bucket}/artisan-tutorial/{uuid4().hex}"
delta_root, staging_root, files_root = (
    f"{prefix}/{name}" for name in ("delta", "staging", "files")
)
runner = ModalRunner(
    environment=Environment.from_pixi(manifest),
    app_name=app_name,
    cache_root=cache_root,
)
fs = storage.filesystem()
read_options = {
    "fs": fs,
    "storage_options": storage.delta_storage_options(),
    "files_root": files_root,
}

## Prepare the selected workers

`WaitTool` is an installed command operation that writes a CSV marker for each
input. Its Bash command runs inside the selected worker; it is not a separate
HTTP tool endpoint. Use the same parameters and resources for preparation and
submission. Local preparation also qualifies the managed command witness.

In [ ]:
from artisan.operations.examples import DataGenerator, WaitTool
from artisan.orchestration import PipelineManager, StepDisposition
from artisan.orchestration.runners import LocalRunner
from artisan.schemas import RunnerResources
from artisan.visualization import inspect_csv, inspect_pipeline, inspect_step

resources = RunnerResources(cpus=1, memory_gb=4, time_limit="00:05:00")
local = LocalRunner(cache_root=cache_root)
local.prepare(WaitTool, params={"seconds": 1}, runner_resources=resources)
runner.prepare(WaitTool, params={"seconds": 1}, runner_resources=resources)

## Submit the same operation twice

All authoritative roots are explicit shared S3 locations. The working directory
belongs to the controller; each remote call owns its own scratch directory.
`skip_cache=True` makes the local and remote observations actual executions.

In [ ]:
with PipelineManager.create(
    name="modal_tutorial",
    delta_root=delta_root,
    staging_root=staging_root,
    files_root=files_root,
    working_root=str(workspace),
    storage=storage,
) as pipeline:
    source = pipeline.run(DataGenerator, params={"count": 2, "seed": 42})
    results = []
    for name, selected_runner in (("local", local), ("modal", runner)):
        result = pipeline.run(
            WaitTool,
            name=name,
            inputs={"dataset": source.output("datasets")},
            params={"seconds": 1},
            step_runner=selected_runner,
            runner_resources=resources,
            skip_cache=True,
            batch_strategy={"artifacts_per_unit": 1, "units_per_worker": 2},
        )
        results.append(result)
    summary = pipeline.finalize()
assert summary["overall_success"]
for result in results:
    assert result.disposition is StepDisposition.EXECUTED
    assert (
        inspect_step(
            delta_root,
            result.step_number,
            pipeline_run_id=pipeline.config.pipeline_run_id,
            **read_options,
        ).height
        == 2
    )
inspect_pipeline(
    delta_root, pipeline_run_id=pipeline.config.pipeline_run_id, **read_options
)

## Inspect invocation markers

Two scientific units were placed in an ordered worker assignment. Each retains
its own accepted result and cancellation boundary. A larger worker pack does not
merge their scientific identities or create child execution rows.

The marker contains the actual task ID or hostname observed by the command. It
is an observation, not a promise about container count or concurrency. Use the
Modal dashboard for startup and provider logs.

In [ ]:
remote = results[1]
markers = inspect_csv(
    delta_root,
    step_number=remote.step_number,
    pipeline_run_id=pipeline.config.pipeline_run_id,
    **read_options,
)
assert markers.height == 2
assert markers["seconds"].to_list() == [1, 1]
assert all(markers["host"].to_list())
print(markers.select("seconds", "host", "source"))

## Cleanup

Run cleanup only after successful finalization and inspection. It closes the
pipeline again before deleting this notebook's unique store prefix and local
working directory. A failed or unresolved run keeps both for diagnosis. Do not
reuse this command on a shared production prefix. Prepared images and named
deployments have separate lifetimes; remove the application created for `app_name`
through Modal after you finish.

In [ ]:
pipeline.close()
assert summary["overall_success"]
if fs.exists(prefix):
    fs.rm(prefix, recursive=True)
shutil.rmtree(workspace)

Continue with [external outputs in object storage](05-modal-r2-outputs.ipynb),
or read [Run Remote Workers](../../how-to-guides/running-remote-workers.md) for
preparation, shared storage, and cancellation requirements.